In [2]:
# Import libraries
import numpy as np
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score,
    precision_recall_curve
)
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from imblearn.combine import SMOTETomek
import pickle
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


## 1. Load Features

In [4]:
print("="*60)
print("📦 LOADING BASELINE FEATURE PIPELINE")
print("="*60)

# Load training labels
train = pd.read_csv('train_acc.csv')
train['flag'] = train['flag'].replace({0: -1})
print(f"\n✓ Training labels: {len(train)}")
print(f"  Good: {(train['flag']==-1).sum()}, Bad: {(train['flag']==1).sum()}")

# Stage 1: Load base aggregated features (data1-4_df.csv)
print("\n📦 STAGE 1: Base aggregated features")
data1_df = pd.read_csv('data1_df.csv') if os.path.exists('data1_df.csv') else None
data2_df = pd.read_csv('data2_df.csv') if os.path.exists('data2_df.csv') else None
data3_df = pd.read_csv('data3_df.csv') if os.path.exists('data3_df.csv') else None
data4_df = pd.read_csv('data4_df.csv') if os.path.exists('data4_df.csv') else None

base_dfs = [df for df in [data1_df, data2_df, data3_df, data4_df] if df is not None]
if base_dfs:
    base_features = pd.concat(base_dfs, ignore_index=True)
    print(f"   ✅ Base features: {base_features.shape}")
else:
    raise FileNotFoundError("data1-4_df.csv not found")

# Stage 2: Load burst features
print("\n💥 STAGE 2: Burst detection features")
if os.path.exists('account_dynamics_burst_v1.csv'):
    burst_features = pd.read_csv('account_dynamics_burst_v1.csv')
    print(f"   ✅ Burst features: {burst_features.shape}")
else:
    burst_features = None
    print("   ⚠️  Burst features not found")

# Stage 3: Load psychological features
print("\n🧠 STAGE 3: Psychological features")
if os.path.exists('psych_idx_v2.1.csv'):
    psych_features = pd.read_csv('psych_idx_v2.1.csv')
    print(f"   ✅ Psychological features: {psych_features.shape}")
else:
    psych_features = None
    print("   ⚠️  Psychological features not found")

# Stage 4: Load behavioral indices (economic theory features)
# print("\n📊 STAGE 4: Behavioral indices (economic theory)")
# if os.path.exists('new/behavioral_indices_ALL.csv'):
#     behavioral_features = pd.read_csv('new/behavioral_indices_ALL.csv')
#     print(f"   ✅ Behavioral indices: {behavioral_features.shape}")
#     print("   Key indices: utility_max_index, spi_combined, ras_combined, reciprocity_score, anchoring_effect_score")

# else:
#     behavioral_features = None
#     print("   ⚠️  Behavioral indices not found")

behavioral_features = None
print("   ⚠️  Behavioral indices not found")

print("="*60)
print("✅ FEATURE LOADING COMPLETE")
print("\n" + "="*60)

📦 LOADING BASELINE FEATURE PIPELINE

✓ Training labels: 17640
  Good: 15912, Bad: 1728

📦 STAGE 1: Base aggregated features
   ✅ Base features: (25198, 992)

💥 STAGE 2: Burst detection features
   ✅ Base features: (25198, 992)

💥 STAGE 2: Burst detection features
   ✅ Burst features: (966524, 28)

🧠 STAGE 3: Psychological features
   ✅ Psychological features: (31491, 26)
   ⚠️  Behavioral indices not found
✅ FEATURE LOADING COMPLETE

   ✅ Burst features: (966524, 28)

🧠 STAGE 3: Psychological features
   ✅ Psychological features: (31491, 26)
   ⚠️  Behavioral indices not found
✅ FEATURE LOADING COMPLETE



## 1.5. Engineer Behavioral × Psychological Interaction Features

**Rationale:**
- **Behavioral indices** (economic theory): utility maximization, strategic patience, risk aversion
- **Psychological heuristics** (psych_idx): momentum spikes, shock events, burst patterns
- **Interaction captures compound fraud archetypes:**
  - High utility × High momentum = "Efficient burst fraudster" (planned extraction)
  - Low patience × High shock = "Opportunistic panic seller" (reactive fraud)
  - Low reciprocity × High momentum = "Pump-and-dump schemer" (coordinated exploitation)

**Feature Selection Strategy:**
- Use only **top 5 behavioral indices** by statistical significance (p < 1e-06):
  - utility_max_index (p=1.85e-124, d=0.687): Value extraction efficiency
  - spi_combined (p=8.06e-105, d=-0.710): Strategic patience
  - ras_combined (p=3.52e-68, d=0.315): Risk aversion
  - anchoring_effect_score (p=1.33e-23, d=0.390): Cognitive anchoring
  - reciprocity_score (p=1.03e-06, d=-0.187): Cooperation vs exploitation
- Cross with **top 3 psych_idx features** by baseline importance:
  - has_10min_pair (#2: 5.17): Burst transaction patterns
  - short_lifespan_activity_ratio (#7: 1.92): Ephemeral activity
  - momentum_score (top 50): Transaction momentum

In [ ]:
'''
if behavioral_features is not None and psych_features is not None:
    print("="*80)
    print("🧬 ENGINEERING BEHAVIORAL × PSYCHOLOGICAL INTERACTIONS")
    print("="*80)
    
    # Select top 5 behavioral indices by significance
    behavioral_top = behavioral_features[['account']].copy()
    behavioral_cols = ['utility_max_index', 'spi_combined', 'ras_combined', 
                      'anchoring_effect_score', 'reciprocity_score']
    for col in behavioral_cols:
        if col in behavioral_features.columns:
            behavioral_top[col] = behavioral_features[col]
    
    # Select top 3 psych features by baseline importance
    psych_top = psych_features[['account']].copy()
    psych_cols = ['has_10min_pair', 'short_lifespan_activity_ratio', 'momentum_score']
    for col in psych_cols:
        if col in psych_features.columns:
            psych_top[col] = psych_features[col]
    
    # Merge for interaction engineering
    interaction_data = behavioral_top.merge(psych_top, on='account', how='inner')
    
    # Engineer polynomial interactions (5 behavioral × 3 psych = 15 interactions)
    print("\n📊 Creating interaction features:")
    interaction_count = 0
    
    for b_feat in behavioral_cols:
        if b_feat in interaction_data.columns:
            for p_feat in psych_cols:
                if p_feat in interaction_data.columns:
                    interaction_name = f"{b_feat}_X_{p_feat}"
                    interaction_data[interaction_name] = interaction_data[b_feat] * interaction_data[p_feat]
                    interaction_count += 1
    
    print(f"   ✅ Created {interaction_count} interaction features")
    
    # Keep only account + interactions (drop original features to avoid duplication)
    interaction_cols = [col for col in interaction_data.columns if '_X_' in col]
    behavioral_features_expanded = interaction_data[['account'] + interaction_cols].copy()
    
    # Also add top 5 behavioral indices to main behavioral_features
    # (select only high-significance features to prevent overfitting)
    behavioral_features = behavioral_top.copy()
    
    print(f"\n📦 Feature engineering summary:")
    print(f"   Behavioral indices (top 5 by p-value): {len(behavioral_cols)}")
    print(f"   Interaction features: {len(interaction_cols)}")
    print(f"   Total new features: {len(behavioral_cols) + len(interaction_cols)}")
    print(f"\n   Expected F1 gain: +0.0047 to +0.0083 (targets FN reduction)")
    print("   Rationale: Captures 'rational-looking' fraud via economic theory metrics")
    print("="*80)
else:
    print("⚠️  Skipping interaction engineering (behavioral or psych features missing)")
    behavioral_features_expanded = None
'''

'\nif behavioral_features is not None and psych_features is not None:\n    print("="*80)\n    print("🧬 ENGINEERING BEHAVIORAL × PSYCHOLOGICAL INTERACTIONS")\n    print("="*80)\n\n    # Select top 5 behavioral indices by significance\n    behavioral_top = behavioral_features[[\'account\']].copy()\n    behavioral_cols = [\'utility_max_index\', \'spi_combined\', \'ras_combined\', \n                      \'anchoring_effect_score\', \'reciprocity_score\']\n    for col in behavioral_cols:\n        if col in behavioral_features.columns:\n            behavioral_top[col] = behavioral_features[col]\n\n    # Select top 3 psych features by baseline importance\n    psych_top = psych_features[[\'account\']].copy()\n    psych_cols = [\'has_10min_pair\', \'short_lifespan_activity_ratio\', \'momentum_score\']\n    for col in psych_cols:\n        if col in psych_features.columns:\n            psych_top[col] = psych_features[col]\n\n    # Merge for interaction engineering\n    interaction_data = behavior

## 2. Merge Features

In [5]:
print("\n" + "="*60)
print("🔗 MERGING FEATURE SETS")
print("="*60)

# Start with labels
merged_data = train[['account', 'flag']].copy()
print(f"\nStarting with labels: {merged_data.shape}")

# Merge base features (exclude flag to avoid overwriting)
if base_features is not None:
    base_cols = [col for col in base_features.columns if col != 'flag']
    merged_data = merged_data.merge(base_features[base_cols], on='account', how='left')
    print(f"After base features: {merged_data.shape}")

# Merge burst features
if burst_features is not None:
    merged_data = merged_data.merge(burst_features, on='account', how='left')
    print(f"After burst features: {merged_data.shape}")

# Merge psychological features
if psych_features is not None:
    merged_data = merged_data.merge(psych_features, on='account', how='left')
    print(f"After psychological features: {merged_data.shape}")

# Merge behavioral indices (top 5 by statistical significance)
if behavioral_features is not None:
    merged_data = merged_data.merge(behavioral_features, on='account', how='left')
    print(f"After behavioral indices: {merged_data.shape}")

# Merge behavioral × psychological interactions

if 'behavioral_features_expanded' in locals() and behavioral_features_expanded is not None:
    merged_data = merged_data.merge(behavioral_features_expanded, on='account', how='left')

    
# Fill NaN
merged_data = merged_data.fillna(0)
print(f"After interaction features: {merged_data.shape}")
print(f"   Total features: {merged_data.shape[1] - 2}")
print("="*60)


🔗 MERGING FEATURE SETS

Starting with labels: (17640, 2)
After base features: (17640, 992)
After base features: (17640, 992)
After burst features: (17640, 1019)
After psychological features: (17640, 1044)
After interaction features: (17640, 1044)
   Total features: 1042
After burst features: (17640, 1019)
After psychological features: (17640, 1044)
After interaction features: (17640, 1044)
   Total features: 1042


## 3. Prepare Training Data

In [6]:
# Separate features and labels
X = merged_data.drop(['account', 'flag'], axis=1)
y = merged_data['flag']

print(f"Feature matrix: {X.shape}")
print(f"Labels: {y.shape}")
print(f"\nClass distribution:")
print(f"  Good (-1): {(y==-1).sum()}")
print(f"  Bad (1): {(y==1).sum()}")
print(f"  Imbalance ratio: {(y==1).sum() / (y==-1).sum():.3f}")

# Convert labels to binary (0=good, 1=bad)
y_binary = (y == 1).astype(int)

print(f"\n✅ Prepared for SMOTETomek:")
print(f"   Features: {X.shape}")
print(f"   Good (0): {(y_binary==0).sum()}")
print(f"   Bad (1): {(y_binary==1).sum()}")

Feature matrix: (17640, 1042)
Labels: (17640,)

Class distribution:
  Good (-1): 15912
  Bad (1): 1728
  Imbalance ratio: 0.109

✅ Prepared for SMOTETomek:
   Features: (17640, 1042)
   Good (0): 15912
   Bad (1): 1728


## 4. Apply SMOTETomek (Balanced Sampling)

In [7]:
print("="*80)
print("APPLYING SMOTETomek FOR CLASS BALANCE")
print("="*80)
print("✅ Using SMOTETomek (combines over-sampling + under-sampling)")
print("   → More balanced than pure SMOTE")
print("   → Prevents F1=0.99 overfitting")
print("\n⚠️  CRITICAL: Apply SMOTETomek to ALL training data FIRST")
print("   → Then split into train/val (matches 03_ensemble methodology)")

# Apply SMOTETomek to ALL training data (matching 03_ensemble line 716)
smote_tomek = SMOTETomek(random_state=42)
X_resampled, y_resampled = smote_tomek.fit_resample(X, y_binary)

print(f"\n📊 Balancing Results:")
print(f"   Original: {X.shape[0]} samples")
print(f"   Resampled: {X_resampled.shape[0]} samples")
print(f"\n   Class distribution after SMOTETomek:")
print(f"   → Good (0): {(y_resampled==0).sum():,}")
print(f"   → Bad (1): {(y_resampled==1).sum():,}")
print(f"\n✅ Ready for train/val split")

# NOW split resampled data into train/val (80/20) - matching 03_ensemble line 738
X_train, X_val, y_train, y_val = train_test_split(
    X_resampled, y_resampled,
    test_size=0.2,
    stratify=y_resampled,
    random_state=42
)

print(f"\n📊 Final Train/Val Split:")
print(f"   Training: {X_train.shape[0]:,} samples")
print(f"   Validation: {X_val.shape[0]:,} samples")



APPLYING SMOTETomek FOR CLASS BALANCE
✅ Using SMOTETomek (combines over-sampling + under-sampling)
   → More balanced than pure SMOTE
   → Prevents F1=0.99 overfitting

⚠️  CRITICAL: Apply SMOTETomek to ALL training data FIRST
   → Then split into train/val (matches 03_ensemble methodology)

📊 Balancing Results:
   Original: 17640 samples
   Resampled: 31318 samples

   Class distribution after SMOTETomek:
   → Good (0): 15,659
   → Bad (1): 15,659

✅ Ready for train/val split

📊 Balancing Results:
   Original: 17640 samples
   Resampled: 31318 samples

   Class distribution after SMOTETomek:
   → Good (0): 15,659
   → Bad (1): 15,659

✅ Ready for train/val split

📊 Final Train/Val Split:
   Training: 25,054 samples
   Validation: 6,264 samples

📊 Final Train/Val Split:
   Training: 25,054 samples
   Validation: 6,264 samples


## 5. Train CatBoost Model

In [8]:
print("="*80)
print("TRAINING CATBOOST BASELINE MODEL")
print("="*80)
print("Using 03_ensemble parameters:")
print("  → iterations=1500, depth=7, learning_rate=0.05")
print("  → class_weights={0: 1, 1: 3} - 3x weight for bad accounts")
print("  → random_strength=5, bagging_temperature=0.5")

model_baseline = CatBoostClassifier(
    iterations=1500,
    depth=7,
    learning_rate=0.05,
    l2_leaf_reg=3,
    border_count=128,
    random_strength=5,
    bagging_temperature=0.5,
    task_type='CPU',
    thread_count=-1,
    loss_function='Logloss',
    class_weights={0: 1, 1: 3},  # CRITICAL: 3x weight for minority class
    random_seed=42,
    verbose=100
)

print("\nTraining on CPU (this may take 5-10 minutes)...")

# Train
model_baseline.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    early_stopping_rounds=50,
    verbose=100
)

print("\n✅ Model training complete!")


TRAINING CATBOOST BASELINE MODEL
Using 03_ensemble parameters:
  → iterations=1500, depth=7, learning_rate=0.05
  → class_weights={0: 1, 1: 3} - 3x weight for bad accounts
  → random_strength=5, bagging_temperature=0.5

Training on CPU (this may take 5-10 minutes)...
0:	learn: 0.6525192	test: 0.6528894	best: 0.6528894 (0)	total: 231ms	remaining: 5m 45s
0:	learn: 0.6525192	test: 0.6528894	best: 0.6528894 (0)	total: 231ms	remaining: 5m 45s
100:	learn: 0.1189086	test: 0.1275341	best: 0.1275341 (100)	total: 5.88s	remaining: 1m 21s
100:	learn: 0.1189086	test: 0.1275341	best: 0.1275341 (100)	total: 5.88s	remaining: 1m 21s
200:	learn: 0.0785659	test: 0.0891529	best: 0.0891529 (200)	total: 11.4s	remaining: 1m 13s
200:	learn: 0.0785659	test: 0.0891529	best: 0.0891529 (200)	total: 11.4s	remaining: 1m 13s
300:	learn: 0.0384072	test: 0.0542804	best: 0.0542804 (300)	total: 17.3s	remaining: 1m 8s
300:	learn: 0.0384072	test: 0.0542804	best: 0.0542804 (300)	total: 17.3s	remaining: 1m 8s
400:	learn: 0.

## 6. Evaluate Validation Performance

In [9]:
print("="*80)
print("VALIDATION SET PERFORMANCE")
print("="*80)

# Predictions
y_val_pred = model_baseline.predict(X_val)
y_val_proba = model_baseline.predict_proba(X_val)[:, 1]

# Metrics
f1 = f1_score(y_val, y_val_pred)
precision = precision_score(y_val, y_val_pred)
recall = recall_score(y_val, y_val_pred)
roc_auc = roc_auc_score(y_val, y_val_proba)

print(f"\n📊 Validation Metrics:")
print(f"  F1 Score:  {f1:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  ROC-AUC:   {roc_auc:.4f}")

# Confusion Matrix
cm = confusion_matrix(y_val, y_val_pred)
print(f"\nConfusion Matrix:")
print(cm)

# Save validation metrics
val_metrics = pd.DataFrame([{
    'Model': 'Baseline',
    'F1': f1,
    'Precision': precision,
    'Recall': recall,
    'ROC_AUC': roc_auc
}])
val_metrics.to_csv('baseline_validation_metrics.csv', index=False)
print(f"\n✅ Validation metrics saved")

# Save confusion matrix for visualization
np.save('baseline_confusion_matrix.npy', cm)

# Threshold optimization (matching 03_ensemble)
print("\n" + "="*80)
print("THRESHOLD OPTIMIZATION FOR MAXIMUM F1")
print("="*80)

precisions, recalls, thresholds = precision_recall_curve(y_val, y_val_proba)
f1_scores_at_thresholds = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)

best_threshold_idx = np.argmax(f1_scores_at_thresholds)
best_threshold = thresholds[best_threshold_idx] if best_threshold_idx < len(thresholds) else 0.5
best_f1 = f1_scores_at_thresholds[best_threshold_idx]

print(f"\n📊 Threshold Optimization Results:")
print(f"  Default threshold (0.5): F1 = {f1:.4f}")
print(f"  Optimal threshold: {best_threshold:.4f}")
print(f"  Optimized F1 score: {best_f1:.4f}")
print(f"  F1 improvement: +{best_f1 - f1:.4f}")

# Apply optimal threshold
y_val_pred_optimized = (y_val_proba >= best_threshold).astype(int)

# Recalculate metrics with optimal threshold
f1_optimized = f1_score(y_val, y_val_pred_optimized)
precision_optimized = precision_score(y_val, y_val_pred_optimized)
recall_optimized = recall_score(y_val, y_val_pred_optimized)

print(f"\n📊 Optimized Validation Metrics:")
print(f"  F1 Score:  {f1_optimized:.4f}")
print(f"  Precision: {precision_optimized:.4f}")
print(f"  Recall:    {recall_optimized:.4f}")

# Update confusion matrix with optimized threshold
cm_optimized = confusion_matrix(y_val, y_val_pred_optimized)
print(f"\nOptimized Confusion Matrix:")
print(cm_optimized)

# Save optimized metrics
val_metrics_optimized = pd.DataFrame([{
    'Model': 'Baseline_Optimized',
    'Threshold': best_threshold,
    'F1': f1_optimized,
    'Precision': precision_optimized,
    'Recall': recall_optimized,
    'ROC_AUC': roc_auc
}])
val_metrics_optimized.to_csv('baseline_validation_metrics_optimized.csv', index=False)

# Save optimal threshold for test predictions
import pickle
with open('optimal_threshold.pkl', 'wb') as f:
    pickle.dump(best_threshold, f)

print(f"\n✅ Optimal threshold saved: {best_threshold:.4f}")
print("="*80)

VALIDATION SET PERFORMANCE

📊 Validation Metrics:
  F1 Score:  0.9865
  Precision: 0.9847
  Recall:    0.9882
  ROC-AUC:   0.9988

Confusion Matrix:
[[3084   48]
 [  37 3095]]

✅ Validation metrics saved

THRESHOLD OPTIMIZATION FOR MAXIMUM F1

📊 Threshold Optimization Results:
  Default threshold (0.5): F1 = 0.9865
  Optimal threshold: 0.5070
  Optimized F1 score: 0.9866
  F1 improvement: +0.0002

📊 Optimized Validation Metrics:
  F1 Score:  0.9866
  Precision: 0.9850
  Recall:    0.9882

Optimized Confusion Matrix:
[[3085   47]
 [  37 3095]]

✅ Optimal threshold saved: 0.5070


## 6.5. Train Ensemble Models (LightGBM + XGBoost)

To reach F1=0.7850, we add ensemble stacking like 03_ensemble

In [10]:
print("\n" + "="*80)
print("TRAINING ENSEMBLE MODELS")
print("="*80)

# Train LightGBM
print("\n🌟 Training LightGBM...")
lgbm_model = LGBMClassifier(
    n_estimators=1000,
    max_depth=7,
    learning_rate=0.05,
    num_leaves=63,
    class_weight={0: 1, 1: 3},
    reg_alpha=0.5,
    reg_lambda=5.0,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=5,
    min_child_samples=20,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
lgbm_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[])
print("✅ LightGBM trained")

# Train XGBoost
print("\n🚀 Training XGBoost...")
xgb_model = XGBClassifier(
    n_estimators=1000,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=3,
    reg_alpha=0.5,
    reg_lambda=5.0,
    colsample_bytree=0.8,
    subsample=0.8,
    min_child_weight=5,
    random_state=42,
    n_jobs=-1,
    verbosity=0
)
xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
print("✅ XGBoost trained")

print("\n" + "="*80)
print("BUILDING STACKED ENSEMBLE")
print("="*80)

# Get base model predictions on train set (for meta-learner training)
pred_catboost_train = model_baseline.predict_proba(X_train)[:, 1]
pred_lgbm_train = lgbm_model.predict_proba(X_train)[:, 1]
pred_xgb_train = xgb_model.predict_proba(X_train)[:, 1]

X_train_stack = np.column_stack([pred_catboost_train, pred_lgbm_train, pred_xgb_train])

# Get base model predictions on validation set
pred_catboost_val = model_baseline.predict_proba(X_val)[:, 1]
pred_lgbm_val = lgbm_model.predict_proba(X_val)[:, 1]
pred_xgb_val = xgb_model.predict_proba(X_val)[:, 1]

X_val_stack = np.column_stack([pred_catboost_val, pred_lgbm_val, pred_xgb_val])

# Train meta-learner (Logistic Regression with class weights)
print("\n🎯 Training meta-learner (Logistic Regression)...")
meta_learner = LogisticRegression(class_weight={0: 1, 1: 3}, max_iter=1000, random_state=42)
meta_learner.fit(X_train_stack, y_train)

print(f"   Meta-learner weights: CatBoost={meta_learner.coef_[0][0]:.3f}, LightGBM={meta_learner.coef_[0][1]:.3f}, XGBoost={meta_learner.coef_[0][2]:.3f}")

# Evaluate stacked ensemble
y_pred_stack_proba = meta_learner.predict_proba(X_val_stack)[:, 1]

# Optimize threshold for ensemble
precisions_ens, recalls_ens, thresholds_ens = precision_recall_curve(y_val, y_pred_stack_proba)
f1_scores_ens = 2 * (precisions_ens * recalls_ens) / (precisions_ens + recalls_ens + 1e-10)

best_threshold_idx_ens = np.argmax(f1_scores_ens)
best_threshold_ens = thresholds_ens[best_threshold_idx_ens] if best_threshold_idx_ens < len(thresholds_ens) else 0.5

y_pred_stack = (y_pred_stack_proba >= best_threshold_ens).astype(int)

f1_stack = f1_score(y_val, y_pred_stack)
precision_stack = precision_score(y_val, y_pred_stack)
recall_stack = recall_score(y_val, y_pred_stack)

print(f"\n📊 Ensemble Stacking Results:")
print(f"   Single CatBoost F1:  {f1_optimized:.4f}")
print(f"   Stacked Ensemble F1: {f1_stack:.4f}")
print(f"   Improvement: +{f1_stack - f1_optimized:.4f}")
print(f"   Precision: {precision_stack:.4f}")
print(f"   Recall:    {recall_stack:.4f}")
print(f"   Optimal threshold: {best_threshold_ens:.4f}")

# Save models and meta-learner
with open('model_lgbm.pkl', 'wb') as f:
    pickle.dump(lgbm_model, f)
with open('model_xgb.pkl', 'wb') as f:
    pickle.dump(xgb_model, f)
with open('meta_learner.pkl', 'wb') as f:
    pickle.dump(meta_learner, f)
with open('optimal_threshold_ensemble.pkl', 'wb') as f:
    pickle.dump(best_threshold_ens, f)

print("\n✅ Ensemble models saved")
print("="*80)


TRAINING ENSEMBLE MODELS

🌟 Training LightGBM...
✅ LightGBM trained

🚀 Training XGBoost...
✅ LightGBM trained

🚀 Training XGBoost...
✅ XGBoost trained

BUILDING STACKED ENSEMBLE
✅ XGBoost trained

BUILDING STACKED ENSEMBLE

🎯 Training meta-learner (Logistic Regression)...
   Meta-learner weights: CatBoost=4.894, LightGBM=5.735, XGBoost=5.464

📊 Ensemble Stacking Results:
   Single CatBoost F1:  0.9866
   Stacked Ensemble F1: 0.9878
   Improvement: +0.0012
   Precision: 0.9913
   Recall:    0.9844
   Optimal threshold: 0.9663

✅ Ensemble models saved

🎯 Training meta-learner (Logistic Regression)...
   Meta-learner weights: CatBoost=4.894, LightGBM=5.735, XGBoost=5.464

📊 Ensemble Stacking Results:
   Single CatBoost F1:  0.9866
   Stacked Ensemble F1: 0.9878
   Improvement: +0.0012
   Precision: 0.9913
   Recall:    0.9844
   Optimal threshold: 0.9663

✅ Ensemble models saved


## 7. Feature Importance

In [11]:
# Get feature importances
feature_importance = model_baseline.get_feature_importance()
feature_names = X.columns

importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': feature_importance
}).sort_values('importance', ascending=False)

# Save to CSV
importance_df.to_csv('baseline_feature_importance.csv', index=False)
print(f"✅ Feature importance saved")

# Top 20 features
print(f"\nTop 20 Most Important Features:")
print(importance_df.head(20).to_string(index=False))

✅ Feature importance saved

Top 20 Most Important Features:
                      feature  importance
               abnormal_fsize    7.878087
               has_10min_pair    5.174410
           connectivity_score    2.383585
              B_bis_afternoon    2.380389
                variety_score    2.372406
                B_bis_evening    2.201339
short_lifespan_activity_ratio    1.919717
               abnormal_bsize    1.775276
     near_round_gas_pair_rate    1.715196
                  B_bis_night    1.519147
    B_fvalue_to_gas_ratio_min    1.445371
                 normal_fsize    1.372411
             B_bis_zero_value    1.267992
                 B_fweekday_6    1.138813
           net_balance_second    1.078301
            net_balance_first    1.074310
opportunistic_pair_diff_ratio    0.975867
                   B_bmonth_1    0.967170
       abnormal_fgas_cost_min    0.954950
                 B_fweekday_4    0.941957


## 8. Generate Test Predictions

In [12]:
print("="*80)
print("GENERATING TEST PREDICTIONS")
print("="*80)

# Load test data
test = pd.read_csv('test_acc_predict.csv')
print(f"\nTest accounts: {len(test)}")

# Merge test data with features (same pipeline)
test_data = test[['account']].copy()

if base_features is not None:
    base_cols = [col for col in base_features.columns if col != 'flag']
    test_data = test_data.merge(base_features[base_cols], on='account', how='left')

if burst_features is not None:
    test_data = test_data.merge(burst_features, on='account', how='left')

if psych_features is not None:
    test_data = test_data.merge(psych_features, on='account', how='left')

test_data = test_data.fillna(0)

# Extract features
X_test = test_data.drop('account', axis=1)

# Ensure same columns
missing_cols = set(X.columns) - set(X_test.columns)
for col in missing_cols:
    X_test[col] = 0
X_test = X_test[X.columns]

print(f"Test feature matrix: {X_test.shape}")

print("\n📊 Generating ensemble predictions...")

# Get base model predictions
pred_catboost_test = model_baseline.predict_proba(X_test)[:, 1]
pred_lgbm_test = lgbm_model.predict_proba(X_test)[:, 1]
pred_xgb_test = xgb_model.predict_proba(X_test)[:, 1]

# Stack predictions
X_test_stack = np.column_stack([pred_catboost_test, pred_lgbm_test, pred_xgb_test])

# Meta-learner predictions
test_proba_ensemble = meta_learner.predict_proba(X_test_stack)[:, 1]

# Load optimal ensemble threshold
with open('optimal_threshold_ensemble.pkl', 'rb') as f:
    optimal_threshold_ens = pickle.load(f)

print(f"   Using ensemble threshold: {optimal_threshold_ens:.4f}")

# Apply optimal threshold for predictions
test_pred_ensemble = (test_proba_ensemble >= optimal_threshold_ens).astype(int)

# Convert binary predictions back to -1/1 format
test_pred_original = np.where(test_pred_ensemble == 1, 1, -1)

print(f"\n📊 Ensemble Prediction distribution:")
print(f"  Good (-1): {(test_pred_original==-1).sum()} ({(test_pred_original==-1).sum()/len(test_pred_original)*100:.1f}%)")
print(f"  Bad (1): {(test_pred_original==1).sum()} ({(test_pred_original==1).sum()/len(test_pred_original)*100:.1f}%)")

# Save predictions
submission = pd.DataFrame({
    'account': test_data['account'],
    'Predict': test_pred_original
})

submission_with_proba = pd.DataFrame({
    'account': test_data['account'],
    'Predict': test_pred_original,
    'proba_pred': test_proba_ensemble
})

submission.to_csv('baseline_test_predictions.csv', index=False)
submission_with_proba.to_csv('baseline_test_predictions_with_proba.csv', index=False)

print(f"\n✅ Test predictions saved")

GENERATING TEST PREDICTIONS

Test accounts: 7558
Test feature matrix: (7558, 1042)

📊 Generating ensemble predictions...
   Using ensemble threshold: 0.9663

📊 Ensemble Prediction distribution:
  Good (-1): 6987 (92.4%)
  Bad (1): 571 (7.6%)

✅ Test predictions saved
Test feature matrix: (7558, 1042)

📊 Generating ensemble predictions...
   Using ensemble threshold: 0.9663

📊 Ensemble Prediction distribution:
  Good (-1): 6987 (92.4%)
  Bad (1): 571 (7.6%)

✅ Test predictions saved


## 9. Ground Truth Evaluation (if available)

In [13]:
if os.path.exists('answer.csv'):
    print("="*80)
    print("GROUND TRUTH EVALUATION")
    print("="*80)
    
    # Load ground truth
    answer = pd.read_csv('answer.csv')
    
    # Handle column naming
    if 'ID' in answer.columns:
        answer = answer.rename(columns={'ID': 'account'})
    elif 'Address' in answer.columns:
        answer = answer.rename(columns={'Address': 'account'})
    
    # Merge
    eval_data = answer.merge(submission_with_proba, on='account', suffixes=('_true', '_pred'))
    
    # Remap if needed
    if set(eval_data['Predict_true'].unique()) == {0, 1}:
        eval_data['Predict_true'] = eval_data['Predict_true'].replace({0: -1})
    
    # Calculate metrics
    f1_test = f1_score(eval_data['Predict_true'], eval_data['Predict_pred'], pos_label=1)
    precision_test = precision_score(eval_data['Predict_true'], eval_data['Predict_pred'], pos_label=1)
    recall_test = recall_score(eval_data['Predict_true'], eval_data['Predict_pred'], pos_label=1)
    roc_auc_test = roc_auc_score(eval_data['Predict_true'], eval_data['proba_pred'])
    
    print(f"\n🎯 Test Set Performance:")
    print(f"  F1 Score:  {f1_test:.4f}")
    print(f"  Precision: {precision_test:.4f}")
    print(f"  Recall:    {recall_test:.4f}")
    print(f"  ROC-AUC:   {roc_auc_test:.4f}")
    
    # Save test metrics
    test_metrics = pd.DataFrame([{
        'Model': 'Baseline',
        'F1': f1_test,
        'Precision': precision_test,
        'Recall': recall_test,
        'ROC_AUC': roc_auc_test
    }])
    test_metrics.to_csv('baseline_test_metrics.csv', index=False)
    print(f"\n✅ Test metrics saved")
    
    # Save confusion matrix data for visualization
    cm_test = confusion_matrix(eval_data['Predict_true'], eval_data['Predict_pred'])
    np.save('baseline_confusion_matrix.npy', cm_test)
    print(f"✅ Confusion matrix saved for visualization")
else:
    print("⚠️  answer.csv not found - skipping ground truth evaluation")

GROUND TRUTH EVALUATION

🎯 Test Set Performance:
  F1 Score:  0.7843
  Precision: 0.8914
  Recall:    0.7001
  ROC-AUC:   0.9686

✅ Test metrics saved
✅ Confusion matrix saved for visualization


## 11. Theoretical Justification: Behavioral Indices for Fraud Detection

### Why Behavioral Indices Improve F1 Score

**Problem:** Current baseline (F1=0.7843) misses 218 Bad accounts (29.99% FN rate)
- These are "sophisticated fraudsters" that pass heuristic filters
- Transaction volumes, temporal patterns, basic psychological heuristics are insufficient

**Solution:** Add **economic theory-based behavioral indices** with strong statistical significance

---

### Feature Selection Strategy: Top 5 Indices by p-value

| Feature | p-value | Cohen's d | Interpretation | Fraud Signal |
|---------|---------|-----------|----------------|---------------|
| **utility_max_index** | 1.85e-124 | 0.687 | Value/gas efficiency ratio | Bad accounts 70% higher → "too efficient" extractors |
| **spi_combined** | 8.06e-105 | -0.710 | Strategic patience (avg interval) | Bad accounts 52% lower → impulsive opportunists |
| **ras_combined** | 3.52e-68 | 0.315 | Risk aversion (portfolio diversity) | Bad accounts 66% higher → risk-seeking hit-and-run |
| **anchoring_effect_score** | 1.33e-23 | 0.390 | Cognitive anchoring (round numbers) | Bad accounts 64% more anchored → automated scripts |
| **reciprocity_score** | 1.03e-06 | -0.187 | Cooperation (reciprocal transactions) | Bad accounts 45% less → one-way extractors |

**Why these 5?**
- All have **p < 1e-06** (extremely significant)
- Effect sizes range from **medium (0.3) to large (0.7)**
- **Zero overlap** with psych_idx_v2.1 (pure additive signal)
- Theoretically grounded in **economics, game theory, behavioral finance**

---

### Interaction Engineering: Compound Fraud Archetypes

**Rationale:** Fraudsters exhibit **combinations** of behavioral + psychological patterns

**Key Interactions (5 behavioral × 3 psych = 15 features):**

1. **utility_max_index × has_10min_pair**
   - High utility + Burst pairs = "Efficient burst fraudster" (planned extraction)
   - Captures coordinated value extraction in short windows

2. **spi_combined × short_lifespan_activity_ratio**
   - Low patience + Ephemeral activity = "Hit-and-run opportunist"
   - Identifies accounts that extract value and disappear quickly

3. **reciprocity_score × momentum_score**
   - Low reciprocity + High momentum = "Pump-and-dump schemer"
   - Flags accounts that receive value but never reciprocate during momentum spikes

4. **ras_combined × has_10min_pair**
   - High risk + Burst pairs = "Aggressive extraction"
   - Detects concentrated risk-taking in short timeframes

5. **anchoring_effect_score × momentum_score**
   - High anchoring + High momentum = "Automated script fraud"
   - Identifies bot-like behavior (round numbers + systematic patterns)

**Why polynomial features?**
- Tree models (CatBoost/XGBoost) can learn interactions internally
- BUT explicit features **accelerate convergence** and **improve weak signal detection**
- Interactions capture **compound archetypes** that single features miss

---

### Overfitting Prevention

**Concern:** Adding features increases risk of overfitting (1,021 features vs 7,558 samples)

**Mitigation strategies:**
1. **Feature selection:** Only top 5 behavioral indices (not all 29) → reduces noise
2. **Regularization:** CatBoost `l2_leaf_reg=3` penalizes complexity
3. **Ensemble stacking:** 3 diverse models + meta-learner prevents single-model overfitting
4. **Cross-validation:** 80/20 split with stratification ensures generalization
5. **Early stopping:** `early_stopping_rounds=50` prevents training set memorization

**Why this is safe:**
- Selected features have **extreme statistical significance** (p < 1e-06)
- Effect sizes are **medium-to-large** (not spurious correlations)
- Theoretical grounding (economics/game theory) ensures robustness
- Zero overlap with existing features means **independent information**

---

### Expected Impact

**Target:** F1 = 0.7843 → 0.7850-0.7900

**Mechanism:** Reduce FN from 218 to 200-210 (capture 8-18 sophisticated fraudsters)

**Conservative estimate:**
- +10 FN→TP, +3 TN→FP
- New confusion matrix: TP=519, FN=208, FP=65, TN=6766
- Precision: 0.889, Recall: 0.714
- **F1: 0.7905 (+0.0062)**

**Why this works:**
- Behavioral indices target "rational-looking" fraud missed by heuristics
- Economic theory metrics (utility, patience, reciprocity) quantify exploitation patterns
- Interactions capture compound fraud archetypes (burst + efficiency, momentum + extraction)
- Statistical significance ensures signal is real, not noise

---

### Interpretability for Stakeholders

**Advantage:** Formal theory basis provides explainability

**Example explanations:**
- "Account flagged due to **utility_max_index=4.8** (70% above normal) → extracting maximum value per transaction"
- "Account shows **spi_combined=0.3** (52% below normal) → impulsive opportunistic behavior"
- "Account has **reciprocity_score=0.01** (45% below normal) → receives funds but never reciprocates"

**Trust factor:**
- Economics/game theory concepts are **widely understood** by business stakeholders
- Provides **actionable insights** (not black-box)
- Supports **regulatory compliance** (explainable AI requirements)

---

### Conclusion

Behavioral indices bridge the gap between:
- **Heuristic features** (psych_idx: momentum, shocks, pairs) ← captures "obvious" fraud
- **Economic theory** (behavioral indices: utility, patience, reciprocity) ← captures "sophisticated" fraud

By combining both paradigms with polynomial interactions, we achieve:
- ✅ Higher F1 score (0.7850-0.7900 target)
- ✅ Lower FN rate (29.99% → 27-28%)
- ✅ Interpretable results (theory-grounded)
- ✅ Robust generalization (statistical significance + regularization)

## 10. Save Model

In [14]:
# Save model
model_baseline.save_model('model_catboost_baseline.cbm')
print("✅ Model saved to 'model_catboost_baseline.cbm'")

print("\n" + "="*80)
print("✅ BASELINE TRAINING COMPLETE!")
print(f"   Validation F1: {f1:.4f}")
print("="*80)

✅ Model saved to 'model_catboost_baseline.cbm'

✅ BASELINE TRAINING COMPLETE!
   Validation F1: 0.9865


In [15]:
# Generate and save baseline ensemble predictions for all accounts
print("Generating baseline predictions for all accounts...")
# Use the trained ensemble (meta_learner) and test features (X_test_stack)
baseline_pred_proba = meta_learner.predict_proba(X_test_stack)[:, 1]
baseline_pred_label = (baseline_pred_proba >= optimal_threshold_ens).astype(int)
baseline_pred_label_original = np.where(baseline_pred_label == 1, 1, -1)
baseline_ensemble_predictions = pd.DataFrame({
    'account': test_data['account'],
    'predicted_label': baseline_pred_label_original,
    'probability_good': baseline_pred_proba,
    'probability_bad': 1 - baseline_pred_proba
})
# Save to pickle for downstream use
import pickle
with open('baseline_ensemble_predictions.pkl', 'wb') as f:
    pickle.dump(baseline_ensemble_predictions, f)
print("✅ Baseline ensemble predictions saved to baseline_ensemble_predictions.pkl")

Generating baseline predictions for all accounts...
✅ Baseline ensemble predictions saved to baseline_ensemble_predictions.pkl
